# Triple-Hybrid RAG — Full Pipeline
논문: *Performance Optimization Study of Hybrid RAG Engine Integrating Multi-Source Knowledge*

**실행 순서**: 위에서 아래로 순서대로 셀 실행

In [ ]:
# ── Step 0: 설치 ──────────────────────────────────────
!pip install -q langchain>=0.1.0 langchain-openai>=0.0.5 langchain-community
!pip install -q faiss-cpu numpy openai
!pip install -q owlready2 neo4j
!pip install -q streamlit plotly koreanize-matplotlib pandas
print('✅ 설치 완료')

In [ ]:
# ── Step 1: GitHub 클론 ───────────────────────────────
!git clone https://github.com/sdw1621/hybrid-rag-comparsion.git
%cd hybrid-rag-comparsion
import sys; sys.path.insert(0, '.')
print('✅ 레포 클론 완료')

In [ ]:
# ── Step 2: API 키 설정 ───────────────────────────────
import os
from google.colab import userdata
try:
    os.environ['OPENAI_API_KEY'] = userdata.get('OPENAI_API_KEY')
    print('✅ API 키 로드 완료 (Colab Secrets)')
except:
    os.environ['OPENAI_API_KEY'] = input('OpenAI API Key: ')

In [ ]:
# ── Step 3: QueryAnalyzer + DWA 검증 ─────────────────
from src import QueryAnalyzer, DWA

analyzer = QueryAnalyzer()
dwa      = DWA(lambda_=0.3)

test_queries = [
    '김철수 교수의 연구 분야는 무엇인가요?',
    '김철수 교수가 소속된 학과의 40세 이하 교수 목록은?',
    'AI 프로젝트 참여 교수 중 컴퓨터공학과 소속은 누구인가요?',
]

for q in test_queries:
    intent = analyzer.analyze(q)
    w      = dwa.compute(intent)
    print(f'Q: {q}')
    print(f'  유형={intent.query_type} | c_e={intent.c_e:.2f} c_r={intent.c_r:.2f} c_c={intent.c_c:.2f}')
    print(f'  {w}')
    print()

In [ ]:
# ── Step 4: TripleHybridRAG 빌드 ──────────────────────
from src import TripleHybridRAG

rag = TripleHybridRAG(lambda_=0.3, top_k=3)
rag.load_university_sample()
rag.build()

In [ ]:
# ── Step 5: 단일 질의 테스트 ──────────────────────────
result = rag.query('김철수 교수가 담당하는 과목은 무엇인가요?')

print(f'답변: {result.answer}')
print(f'응답시간: {result.elapsed:.3f}s')
print(f'가중치: {result.weights}')
print(f'질의유형: {result.intent.query_type}')
print(f'\nVector 컨텍스트:')
for c in result.vector_contexts: print(f'  - {c[:80]}')
print(f'\nGraph 컨텍스트:')
for c in result.graph_contexts:  print(f'  - {c}')
print(f'\nOntology 컨텍스트:')
for c in result.onto_contexts:   print(f'  - {c}')

In [ ]:
# ── Step 6: Gold QA 5,000쌍 로드 ─────────────────────
import json, random
from data.dataset_generator import save_dataset

# 이미 생성된 파일이 있으면 재사용, 없으면 생성
try:
    with open('data/gold_qa_5000.json', encoding='utf-8') as f:
        full_ds = json.load(f)
    print(f'✅ 기존 파일 로드: {len(full_ds)}쌍')
except FileNotFoundError:
    full_ds = save_dataset('data/gold_qa_5000.json', total=5000)
    print(f'✅ 새로 생성: {len(full_ds)}쌍')

simple_ds      = [d for d in full_ds if d['type'] == 'simple']
multihop_ds    = [d for d in full_ds if d['type'] == 'multi_hop']
conditional_ds = [d for d in full_ds if d['type'] == 'conditional']
print(f'  Simple {len(simple_ds)} / Multi-hop {len(multihop_ds)} / Conditional {len(conditional_ds)}')

In [ ]:
# ── Step 7: 5개 시스템 베이스라인 빌드 ───────────────
import os
from langchain_openai import ChatOpenAI
from src.vector_store import VectorStore
from src.query_analyzer import QueryAnalyzer

llm = ChatOpenAI(model='gpt-4o-mini', temperature=0.0)

# 공통 Vector Store (Vector-Only / GraphRAG / HybridRAG / Adaptive-RAG 공유)
vo_store = VectorStore()
vo_store.build(rag._documents)

PROMPT = (
    '다음 컨텍스트를 기반으로 질문에 정확하게 답변하세요. '
    '컨텍스트에서 답을 찾을 수 없으면 정보를 찾을 수 없습니다라고 답하세요.\n\n'
    'Context:\n{context}\n\nQuestion: {query}\n\nAnswer:'
)

def vector_only_query(q):
    docs = [d for d, _ in vo_store.search(q, 3)]
    resp = llm.invoke(PROMPT.format(context='\n'.join(docs), query=q))
    return resp.content, docs, []

def graphrag_query(q):
    g = rag.graph.search(q, 3)
    v = [d for d, _ in vo_store.search(q, 1)]
    resp = llm.invoke(PROMPT.format(context='\n'.join(v + g), query=q))
    return resp.content, v, g

def hybridrag_query(q):
    v = [d for d, _ in vo_store.search(q, 2)]
    g = rag.graph.search(q, 1)
    resp = llm.invoke(PROMPT.format(context='\n'.join(v + g), query=q))
    return resp.content, v, g

def adaptive_query(q):
    intent = QueryAnalyzer().analyze(q)
    if intent.query_type == 'simple':
        docs = [d for d, _ in vo_store.search(q, 3)]
        resp = llm.invoke(PROMPT.format(context='\n'.join(docs), query=q))
        return resp.content, docs, []
    elif intent.query_type == 'multi_hop':
        g = rag.graph.search(q, 3)
        resp = llm.invoke(PROMPT.format(context='\n'.join(g), query=q))
        return resp.content, [], g
    else:
        o = rag.ontology.search(q, 3)
        resp = llm.invoke(PROMPT.format(context='\n'.join(o), query=q))
        return resp.content, [], o

SYSTEMS = {
    'Vector-Only':   vector_only_query,
    'GraphRAG':      graphrag_query,
    'HybridRAG':     hybridrag_query,
    'Adaptive-RAG':  adaptive_query,
    'Triple-Hybrid': None,   # rag.query() 직접 사용
}
print('✅ 5개 시스템 준비 완료')

In [ ]:
# ── Step 8: Table 13 — 전체 5,000 QA × 3runs 실험 ───
# ⚠️  전체 실행 시 API 호출 75,000회 / 예상 비용 $10~15 / 소요 3~5시간
# 샘플만 원하면: sample_ds = random.sample(full_ds, 500)
import time, numpy as np
from src.evaluator import Evaluator

RUNS = 3
random.seed(42)
sample_ds = full_ds          # ← 전체 5,000개 (논문 최종 수치용)
# sample_ds = random.sample(full_ds, 500)   # ← 빠른 테스트용

ev = Evaluator()
exp_results = {}
total_calls = len(sample_ds) * RUNS * len(SYSTEMS)
call_cnt = 0
t0 = time.time()

print(f'실험 시작: {len(sample_ds)}개 × {len(SYSTEMS)}시스템 × {RUNS}회 = {total_calls:,}회 API 호출')
print('=' * 65)

for sys_name, query_fn in SYSTEMS.items():
    print(f'\n▶ [{sys_name}] 평가 중...')
    f1s, ems, r3s, precs, faiths = [], [], [], [], []
    by_type = {'simple': [], 'multi_hop': [], 'conditional': []}

    for i, item in enumerate(sample_ds):
        q, gold, qtype = item['query'], item['answer'], item['type']
        run_ems = []

        for _ in range(RUNS):
            try:
                if sys_name == 'Triple-Hybrid':
                    res = rag.query(q)
                    answer   = res.answer
                    retrieved = res.vector_contexts + res.graph_contexts + res.onto_contexts
                    contexts  = res.vector_contexts
                else:
                    answer, v_docs, g_docs = query_fn(q)
                    retrieved = v_docs + g_docs
                    contexts  = v_docs

                r = ev.evaluate_single(answer, gold, retrieved, contexts)
                run_ems.append(r.em_norm)
                f1s.append(r.f1)
                r3s.append(r.recall_at_k)
                precs.append(r.precision)
                faiths.append(r.faithfulness)

            except Exception as e:
                print(f'\n  ⚠️ 오류: {e}')
                run_ems.append(0.0); f1s.append(0.0)
                r3s.append(0.0); precs.append(0.0); faiths.append(0.0)

            call_cnt += 1

        em_mean = np.mean(run_ems)
        ems.append(em_mean)
        by_type[qtype].append(em_mean)

        # 진행률
        elapsed = time.time() - t0
        eta = (elapsed / call_cnt * (total_calls - call_cnt)) if call_cnt else 0
        print(f'  [{i+1}/{len(sample_ds)}] {call_cnt/total_calls*100:.1f}%  '
              f'경과={elapsed/60:.1f}분  남은={eta/60:.1f}분  현재EM={em_mean:.3f}', end='\r')

    exp_results[sys_name] = {
        'F1':           {'mean': float(np.mean(f1s)),    'std': float(np.std(f1s))},
        'EM':           {'mean': float(np.mean(ems)),    'std': float(np.std(ems))},
        'Recall@3':     {'mean': float(np.mean(r3s)),    'std': float(np.std(r3s))},
        'Precision':    {'mean': float(np.mean(precs)),  'std': float(np.std(precs))},
        'Faithfulness': {'mean': float(np.mean(faiths)), 'std': float(np.std(faiths))},
        'by_type': {
            t: {'mean': float(np.mean(v)), 'std': float(np.std(v))}
            for t, v in by_type.items() if v
        },
    }
    r = exp_results[sys_name]
    print(f'\n  ✅ F1={r["F1"]["mean"]:.4f}±{r["F1"]["std"]:.4f}  '
          f'EM={r["EM"]["mean"]:.4f}±{r["EM"]["std"]:.4f}')

# 결과 저장
import json as _json
out = {'meta': {'sample_size': len(sample_ds), 'runs': RUNS,
                'total_api_calls': call_cnt,
                'elapsed_min': round((time.time()-t0)/60, 1)},
       'results': exp_results}
with open('data/experiment_results_5000qa.json', 'w', encoding='utf-8') as f:
    _json.dump(out, f, ensure_ascii=False, indent=2)

print(f'\n✅ 결과 저장 완료: data/experiment_results_5000qa.json')
print(f'⏱  총 소요: {(time.time()-t0)/60:.1f}분  |  API 호출: {call_cnt:,}회')

In [ ]:
# ── Step 9: Table 14 — 질의 유형별 EM 출력 ───────────
print('=== Table 14. 질의 유형별 EM (논문용) ===')
print(f'{"유형":<14} {"V-Only Raw":>12} {"Triple Raw":>12} {"Δ(%)":>10}')
print('-' * 52)
for t in ['simple', 'multi_hop', 'conditional']:
    vo_em = exp_results['Vector-Only']['by_type'].get(t, {}).get('mean', 0)
    th_em = exp_results['Triple-Hybrid']['by_type'].get(t, {}).get('mean', 0)
    delta = (th_em - vo_em) / max(vo_em, 1e-9) * 100
    print(f'{t:<14} {vo_em:>12.4f} {th_em:>12.4f} {delta:>+10.1f}%')

print('\n=== Table 13. 전체 성능 비교 ===')
print(f'{"시스템":<16} {"F1":>10} {"EM":>10} {"Recall@3":>10} {"Precision":>10} {"Faith":>8}')
print('-' * 68)
for sname in ['Vector-Only','GraphRAG','HybridRAG','Adaptive-RAG','Triple-Hybrid']:
    r = exp_results[sname]
    print(f'{sname:<16} '
          f'{r["F1"]["mean"]:.2f}±{r["F1"]["std"]:.2f}  '
          f'{r["EM"]["mean"]:.2f}±{r["EM"]["std"]:.2f}  '
          f'{r["Recall@3"]["mean"]:.2f}±{r["Recall@3"]["std"]:.2f}  '
          f'{r["Precision"]["mean"]:.2f}±{r["Precision"]["std"]:.2f}  '
          f'{r["Faithfulness"]["mean"]:.2f}±{r["Faithfulness"]["std"]:.2f}')

In [ ]:
# ── Step 10: Table 15 — Ablation Study ───────────────
from src.dwa import DWA
from src.evaluator import Evaluator

random.seed(42)
abl_sample = random.sample(full_ds, 300)   # Ablation은 300개 × 3회
ev2 = Evaluator()

ablation_configs = {
    '(A) Equal Weight':  {'mode': 'equal'},
    '(B) Type-Fixed':    {'mode': 'type_fixed'},
    '(C) Full DWA':      {'mode': 'full'},
}

abl_results = {}
print('=== Table 15. Ablation Study ===')

for cfg_name, cfg in ablation_configs.items():
    print(f'\n  ▶ {cfg_name}')
    f1s, ems = [], []
    by_type = {'simple': [], 'multi_hop': [], 'conditional': []}

    for item in abl_sample:
        q, gold, qtype = item['query'], item['answer'], item['type']
        run_ems = []

        for _ in range(3):
            intent = rag.analyzer.analyze(q)

            if cfg['mode'] == 'equal':
                from src.dwa import DWAWeights
                w = DWAWeights(alpha=0.333, beta=0.333, gamma=0.334)
            elif cfg['mode'] == 'type_fixed':
                base = {'simple':(0.6,0.2,0.2),'multi_hop':(0.2,0.6,0.2),'conditional':(0.2,0.2,0.6)}
                a,b,g = base.get(intent.query_type,(0.4,0.3,0.3))
                from src.dwa import DWAWeights
                w = DWAWeights(alpha=a, beta=b, gamma=g)
            else:
                w = rag.dwa.compute(intent)

            v_ctxs = [d for d, _ in rag.vector.search(q, rag.top_k)]
            g_ctxs = rag.graph.search(q, rag.top_k)
            o_ctxs = rag.ontology.search(q, rag.top_k)
            ctx    = rag._merge_contexts(v_ctxs, g_ctxs, o_ctxs, w.alpha, w.beta, w.gamma)
            from src.triple_hybrid_rag import PROMPT_TEMPLATE
            resp   = rag.llm.invoke(PROMPT_TEMPLATE.format(context=ctx, query=q))
            answer = resp.content

            r = ev2.evaluate_single(answer, gold, v_ctxs+g_ctxs+o_ctxs, v_ctxs)
            run_ems.append(r.em_norm)
            f1s.append(r.f1)

        em_mean = np.mean(run_ems)
        ems.append(em_mean)
        by_type[qtype].append(em_mean)

    abl_results[cfg_name] = {
        'F1': float(np.mean(f1s)), 'F1_std': float(np.std(f1s)),
        'EM': float(np.mean(ems)), 'EM_std': float(np.std(ems)),
        'multi_hop_EM': float(np.mean(by_type['multi_hop'])),
        'conditional_EM': float(np.mean(by_type['conditional'])),
    }
    r = abl_results[cfg_name]
    print(f'    F1={r["F1"]:.4f}±{r["F1_std"]:.4f}  EM={r["EM"]:.4f}  '
          f'Multi-hop EM={r["multi_hop_EM"]:.4f}  Cond EM={r["conditional_EM"]:.4f}')

baseline_f1 = abl_results['(C) Full DWA']['F1']
print('\n  ΔF1 vs (C):')
for k, v in abl_results.items():
    delta = (v['F1'] - baseline_f1) / baseline_f1 * 100
    print(f'    {k}: {delta:+.1f}%')